# A3 独立评估

训练参照 ANOVA 选区、PCA、30近邻 miLISI；绘制各方法分别拟合的真实／合成联合 UMAP。

使用 simscDNAm 环境的 Jupyter，修改 `evaluate/config.yaml` 后从上到下运行。本入口只执行本实验，结果写入配置输出目录的 `A3/`，图写入其 `figures/`。

In [ ]:
from pathlib import Path
PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "model/src/model.py").is_file())
CONFIG_PATH = PROJECT / "evaluate/config.yaml"
EXPERIMENT = "A3"

## 读取配置与指标

相对路径以配置文件所在目录为准；各实验使用同一随机种子和同一批抽样细胞。

In [ ]:
from contextlib import ExitStack
import importlib.metadata
import json
import sys

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.decomposition import PCA
from threadpoolctl import threadpool_limits
import umap
import yaml

PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "model/src/model.py").is_file())
sys.path.insert(0, str(PROJECT))
from data.evaluation import (dense_block, read_labels, region_pairs, validate_real_duplicates,
    natural_reference_statistics, select_features, imputed_features, validate_region_axis, sample_rows)
if Path(sys.prefix).name.lower() != "simscdnam":
    raise RuntimeError(f"请切换到 simscDNAm 的 Jupyter 内核，当前为 {sys.executable}")
CONFIG_PATH = CONFIG_PATH.expanduser().resolve()
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
if not isinstance(config, dict):
    raise ValueError("配置必须是 YAML 映射")
required = {"real_h5ad", "synthetic_h5ad", "output_dir"}
if not required.issubset(config) or not isinstance(config["synthetic_h5ad"], dict) or not config["synthetic_h5ad"]:
    raise ValueError("请填写 real_h5ad、非空 synthetic_h5ad 方法映射和 output_dir")
defaults = {"reference_h5ad": None, "region_pairs_csv": None, "label_key": "Subclass",
    "duplicate_count": 1000, "seed": 42, "evaluation_cells": None, "a3_top_regions": 50000,
    "pca_components": 50, "milisi_neighbors": 30, "umap_seed": 2022, "spearman_top_regions": 2000,
    "a1_regions_csv": None, "genome_assembly": None, "read_in_memory": False}
config = {**defaults, **config}
if not isinstance(config["read_in_memory"], bool):
    raise ValueError("read_in_memory必须为布尔值")
if config["label_key"] != "Subclass":
    raise ValueError("本流程统一使用真实类型 Subclass，请将真实标签字段改名为 Subclass")
if config["duplicate_count"] != 1000:
    raise ValueError("本流程固定最后 1000 个区间为复制区间")
for key in ("a3_top_regions", "pca_components", "milisi_neighbors", "spearman_top_regions"):
    if not isinstance(config[key], int) or isinstance(config[key], bool) or config[key] < 2:
        raise ValueError(f"{key} 必须为不小于 2 的整数")
if config["evaluation_cells"] is not None and (not isinstance(config["evaluation_cells"], int)
        or isinstance(config["evaluation_cells"], bool) or config["evaluation_cells"] < 3):
    raise ValueError("evaluation_cells 必须为 null 或不小于 3 的整数")
if any(not isinstance(name, str) or not name.strip() or name in {"真实数据", "训练参照", "CVAEKMeansAligned"}
       for name in config["synthetic_h5ad"]):
    raise ValueError("方法名须为非空字符串，且不能使用保留名称或暂不参与的 CVAEKMeansAligned")

def resolve_path(value):
    path = Path(value).expanduser()
    return (CONFIG_PATH.parent / path).resolve() if not path.is_absolute() else path.resolve()

plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans", "pdf.fonttype": 42,
                     "axes.spines.top": False, "axes.spines.right": False})
print("Jupyter 内核：", sys.executable)
print("配置文件：", CONFIG_PATH)
from evaluate.A3.metrics import milisi


## 本实验绘图

图中文字使用英文，缺失值和超出 0–1 的值按原协议处理。

In [ ]:
from matplotlib.colors import to_rgba
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator


PLOT_COLORS = {"真实数据": "#A6A6A6", "Real": "#A6A6A6",
    "simscDNAm": "#3B73B9", "CVAE_training_mask": "#3B73B9", "FittedMarginal": "#269D91",
    "simATAC": "#A487C9", "simCAS": "#D95F02", "STADiffuser": "#B44C76"}


def plot_name(name):
    return {"真实数据": "Real", "CVAE_training_mask": "CVAE (training mask)"}.get(name, name)


def plot_color(name):
    return PLOT_COLORS.get(name, "#3B73B9")


def style_axis(axis):
    axis.spines[["top", "right"]].set_visible(False)
    axis.spines[["left", "bottom"]].set_linewidth(0.7)
    axis.grid(axis="y", color="#E8E8E8", linewidth=0.5)
    axis.set_axisbelow(True)


def save_figure(figure, output, stem):
    with plt.rc_context({"pdf.fonttype": 42}):
        for extension in ("png", "pdf"):
            figure.savefig(output / "figures" / f"{stem}.{extension}", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(figure)


def plot_joint_umap(coordinates, summary, methods, output, seed):
    """可直接用本轮正式坐标重绘；图上miLISI来自正式PCA空间得分。"""
    figure, axes = plt.subplots(1, len(methods), figsize=(4 * len(methods), 4.4), squeeze=False)
    figure.subplots_adjust(left=0.035, right=0.995, bottom=0.16, top=0.80, wspace=0.15)
    scores = summary.set_index("方法")["miLISI"]
    for axis, method in zip(axes.ravel(), methods):
        local = coordinates.loc[coordinates["方法"].eq(method)]
        xy = local[["UMAP1", "UMAP2"]].to_numpy()
        source = local["来源"].to_numpy()
        draw = np.random.default_rng(seed).permutation(len(local))
        for name, label, color in [("真实", "Real", "#A6A6A6"), ("合成", "Synthetic", plot_color(method))]:
            keep = draw[source[draw] == name]
            axis.scatter(xy[keep, 0], xy[keep, 1], s=8, alpha=0.65, color=color, linewidths=0, label=label, rasterized=True)
        axis.set(title=f"{plot_name(method)}\nmiLISI = {scores.loc[method]:.4f}", xlabel="UMAP 1", ylabel="UMAP 2", xticks=[], yticks=[])
        axis.title.set_color(plot_color(method))
        axis.title.set_fontsize(10)
        axis.legend(frameon=False, fontsize=8, markerscale=2)
        axis.spines[["top", "right"]].set_visible(False)
    figure.suptitle(f"Real / synthetic mixing (k = {int(summary['近邻数'].iloc[0])})", fontsize=12, weight="bold", y=0.98)
    figure.text(0.035, 0.025, "Each panel has its own joint UMAP in real-reference PCA space. Coordinates are not comparable across panels.\n"
                "miLISI is computed in PCA space, not from these 2D projections.", fontsize=7, color="#555555")
    save_figure(figure, output, "A3_joint_UMAP")


def run_a3(datasets, rows, methods, fill, scores, output, config):
    selected = select_features(scores, config["a3_top_regions"])
    n_components = min(config["pca_components"], len(rows["训练参照"]) - 1, len(selected))
    if n_components < 2 or len(rows["训练参照"]) < 4:
        raise ValueError("真实参照不足以拟合至少二维的 PCA 和 UMAP")
    pd.DataFrame({"原region索引": selected, "原region": datasets["训练参照"].var_names[selected],
                  "ANOVA": scores[selected], "填补中位数": fill[selected]}).to_csv(output / "A3_selected_regions.csv", index=False)
    np.save(output / "reference_natural_medians.npy", fill)
    pca = PCA(n_components=n_components, svd_solver="randomized", random_state=config["seed"])
    representations = {"训练参照": pca.fit_transform(imputed_features(
        datasets["训练参照"], rows["训练参照"], selected, fill))}
    names = ["训练参照", "真实数据", *methods]
    labels = {name: read_labels(datasets[name], rows[name]) for name in names}
    for name in names[1:]:
        if datasets[name] is datasets["训练参照"] and np.array_equal(rows[name], rows["训练参照"]):
            representations[name] = representations["训练参照"]
        else:
            representations[name] = pca.transform(imputed_features(datasets[name], rows[name], selected, fill))
    pc_tables = []
    for name in names:
        frame = pd.DataFrame(representations[name], columns=[f"PC{i + 1}" for i in range(n_components)])
        frame.insert(0, "细胞类型", labels[name])
        frame.insert(0, "细胞ID", datasets[name].obs_names[rows[name]])
        frame.insert(0, "数据集", name)
        pc_tables.append(frame)
    pd.concat(pc_tables, ignore_index=True).to_csv(output / "A3_PCA_coordinates.csv", index=False)
    summaries, cell_tables, group_tables = [], [], []
    for method in methods:
        score, cells, groups = milisi(representations["真实数据"], labels["真实数据"],
            representations[method], labels[method], config["milisi_neighbors"])
        cells.insert(0, "细胞ID", np.r_[datasets["真实数据"].obs_names[rows["真实数据"]],
                                      datasets[method].obs_names[rows[method]]])
        cells.insert(0, "方法", method)
        groups.insert(0, "方法", method)
        cell_tables.append(cells)
        group_tables.append(groups)
        summaries.append({"方法": method, "miLISI": score, "近邻数": config["milisi_neighbors"],
                          "入选区域数": len(selected), "PCA维数": n_components})
    summary = pd.DataFrame(summaries)
    summary.to_csv(output / "A3_miLISI.csv", index=False)
    pd.concat(cell_tables, ignore_index=True).to_csv(output / "A3_miLISI_cells.csv", index=False)
    pd.concat(group_tables, ignore_index=True).to_csv(output / "A3_type_details.csv", index=False)
    print("A3 的 PCA 和 miLISI 已计算，开始绘制 UMAP。", flush=True)
    parameters = dict(n_neighbors=min(config["milisi_neighbors"], len(rows["训练参照"]) - 1),
                      random_state=config["umap_seed"], transform_seed=config["umap_seed"], n_jobs=1)
    joint_tables = []
    for method in methods:
        combined = np.vstack([representations["真实数据"], representations[method]])
        joint_parameters = {**parameters, "n_neighbors": min(config["milisi_neighbors"], len(combined) - 1)}
        xy = umap.UMAP(**joint_parameters).fit_transform(combined)
        source = np.r_[np.repeat("Real", len(rows["真实数据"])), np.repeat("Synthetic", len(rows[method]))]
        joint_tables.append(pd.DataFrame({"方法": method, "来源": np.where(source == "Real", "真实", "合成"),
            "细胞ID": np.r_[datasets["真实数据"].obs_names[rows["真实数据"]], datasets[method].obs_names[rows[method]]],
            "UMAP1": xy[:, 0], "UMAP2": xy[:, 1]}))
    coordinates = pd.concat(joint_tables, ignore_index=True)
    coordinates.to_csv(output / "A3_joint_coordinates.csv", index=False)
    plot_joint_umap(coordinates, summary, methods, output, config["umap_seed"])
    return summary, {"入选区域数": len(selected), "PCA维数": n_components,
                     "解释方差比合计": float(pca.explained_variance_ratio_.sum())}

## 执行与导出

校验完整区域轴及真实复制列；独立训练参照与真实留出细胞不得重叠。失败写入运行记录，读取对象在结束时关闭。

In [ ]:
def evaluate_experiment(config):
    methods = list(config["synthetic_h5ad"])
    paths = {"真实数据": resolve_path(config["real_h5ad"]),
             **{name: resolve_path(value) for name, value in config["synthetic_h5ad"].items()}}
    reference_path = resolve_path(config["reference_h5ad"]) if config["reference_h5ad"] else paths["真实数据"]
    pair_path = resolve_path(config["region_pairs_csv"]) if config["region_pairs_csv"] else None
    region_path = resolve_path(config["a1_regions_csv"]) if config.get("a1_regions_csv") else None
    output = resolve_path(config["output_dir"]) / EXPERIMENT
    if sys.platform != "win32":
        output.relative_to(Path("/home/sccasimp/data/xb/outputs/simscDNAm").resolve())
    all_paths = set(paths.values()) | {reference_path}
    if pair_path:
        all_paths.add(pair_path)
    if EXPERIMENT == "A1" and region_path:
        all_paths.add(region_path)
    if any(path.is_relative_to(output) for path in all_paths):
        raise ValueError("输出目录不能包含输入文件，以免覆盖输入；请选择独立的评估输出目录")
    for path in all_paths:
        if not path.is_file():
            raise FileNotFoundError(f"缺少输入文件：{path}")
    if len(set(paths.values())) != len(paths):
        raise ValueError("真实数据和各方法合成数据必须指向不同文件")
    records = [{"路径": str(path), "字节数": path.stat().st_size, "修改时间纳秒": path.stat().st_mtime_ns}
               for path in sorted(all_paths)]
    output.mkdir(parents=True, exist_ok=True)
    (output / "figures").mkdir(exist_ok=True)
    report = {"状态": "运行中", "标签字段": "Subclass", "最后复制区间数": 1000, "方法": methods,
              "参照模式": "独立训练真实参照" if reference_path != paths["真实数据"] else "评估真实数据拟合参照，非独立留出评估",
              "配置": config, "配置路径": str(CONFIG_PATH), "输入文件": records, "内核": sys.executable,
              "依赖版本": {name: importlib.metadata.version(name) for name in
                  ["numpy", "pandas", "scipy", "anndata", "scikit-learn", "umap-learn"]}}
    def write_report():
        (output / "run_summary.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    write_report()
    try:
        with ExitStack() as stack, threadpool_limits(limits=8):
            datasets = {}
            for name, path in paths.items():
                item = ad.read_h5ad(path, backed=None if config.get("read_in_memory", False) else "r")
                stack.callback(item.file.close)
                datasets[name] = item
            real = datasets["真实数据"]
            if reference_path == paths["真实数据"]:
                datasets["训练参照"] = real
            else:
                item = ad.read_h5ad(reference_path, backed=None if config.get("read_in_memory", False) else "r")
                stack.callback(item.file.close)
                datasets["训练参照"] = item
                if len(real.obs_names.intersection(item.obs_names)):
                    raise ValueError("独立训练参照与评估真实细胞 ID 重叠，请提供分离后的数据")
            for name, item in datasets.items():
                validate_region_axis(real, item)
                if not item.obs_names.is_unique:
                    raise ValueError(f"{name} 的细胞 ID 不唯一")
            pairs = region_pairs(real, pair_path=pair_path)
            validate_real_duplicates(real, pairs)
            if datasets["训练参照"] is not real:
                validate_real_duplicates(datasets["训练参照"], pairs)
            natural_count = real.n_vars - 1000
            count = real.n_obs if config["evaluation_cells"] is None else config["evaluation_cells"]
            rows = {name: sample_rows(item, count, config["seed"]) for name, item in datasets.items() if name != "训练参照"}
            rows["训练参照"] = np.arange(datasets["训练参照"].n_obs)
            labels = {name: read_labels(item, rows[name]) for name, item in datasets.items()}
            if EXPERIMENT == "A3" and 2 * count <= config["milisi_neighbors"]:
                raise ValueError("合并细胞数必须大于 miLISI 近邻数")
            pairs.to_csv(output / "region_pairs.csv", index=False)
            pd.concat([pd.DataFrame({"数据集": name, "原行索引": selected,
                "细胞ID": datasets[name].obs_names[selected], "细胞类型": labels[name]})
                for name, selected in rows.items()], ignore_index=True).to_csv(output / "selected_cells.csv", index=False)
            report.update({"评估细胞数": count, "真实参照细胞数": len(rows["训练参照"]), "原始区间数": natural_count,
                           "总区间数": real.n_vars, "A1_A3排除末尾复制列": True})
            write_report()
            print(f"输入校验通过：原始区间 {natural_count}，复制区间 1000，每种方法评估 {count} 个细胞。", flush=True)

            print("开始 A3：真实训练参照 ANOVA、PCA、30近邻 miLISI 和 UMAP。", flush=True)
            fill, scores = natural_reference_statistics(datasets["训练参照"], rows["训练参照"], natural_count)
            a3, a3_protocol = run_a3(datasets, rows, methods, fill, scores, output, config)
            report["A3"] = a3_protocol
            summary = a3
            diagnostics = []
            for name in ["真实数据", *methods]:
                for domain, start, stop in [("原始区间", 0, natural_count), ("复制区间", natural_count, real.n_vars)]:
                    valid, outside = 0, 0
                    for left in range(start, stop, 512):
                        block = dense_block(datasets[name], rows[name], slice(left, min(left + 512, stop)))
                        finite = np.isfinite(block)
                        valid += int(finite.sum())
                        outside += int((finite & ((block < 0) | (block > 1))).sum())
                    diagnostics.append({"数据集": name, "区间范围": domain, "有限观测数": valid,
                        "缺失比例": 1 - valid / (len(rows[name]) * (stop - start)),
                        "超出0到1比例": outside / valid if valid else np.nan})
            pd.DataFrame(diagnostics).to_csv(output / "input_diagnostics.csv", index=False)
            for record in records:
                path = Path(record["路径"])
                if path.stat().st_size != record["字节数"] or path.stat().st_mtime_ns != record["修改时间纳秒"]:
                    raise RuntimeError(f"评估期间输入文件发生变化：{path}")
            report.update({"状态": "完成", "输入文件大小及修改时间未变": True,
                           "输出目录": str(output), "输出文件": sorted(str(p.relative_to(output)) for p in output.rglob("*") if p.is_file())})
            write_report()
            display(summary)
            print(EXPERIMENT + " 评估完成，结果目录：", output)
            return summary
    except Exception as error:
        report.update({"状态": "失败", "错误": f"{type(error).__name__}: {error}"})
        write_report()
        raise


summary = evaluate_experiment(config)